In [ ]:
# importeren om te kunnen werken
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import math as math
import ewatercycle
import ewatercycle.models
import ewatercycle.forcing

shape_file_area = 7.629080e+03 # in km^2

In [ ]:
#Route naar oppervlakte

opp_file = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "GWSC_dams_mogwai_Brokopondo.csv"
opp = pd.read_csv(opp_file, delimiter=',', skiprows = 1)
opp = opp.rename(columns={"Time (UTC)": "Date"})
opp["Date"] = pd.to_datetime(opp["Date"], utc = True) #opp werkt in "Date" en "Value voor de datum en de km^2

# plt.figure(figsize=(12, 5))
# plt.plot(opp["Date"], opp["Value"])
# plt.grid(); 

In [ ]:
# Hoogtemetingen plotten samen met kritische en zeer kritische hoogte

plt.figure(figsize=(16, 7))

height_file = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "Dahiti Brokopondo water height.xlsx"

# Read malformed Excel
height = pd.read_excel(height_file)

# Split the single column into multiple columns
height = height.iloc[:, 0].str.split(';', expand=True)

# Rename columns
height.columns = ["datetime", "wse", "wse_u"]

# Convert data types
height["datetime"] = pd.to_datetime(height["datetime"], utc=True)
height["wse"] = height["wse"].astype(float)
height["wse_u"] = height["wse_u"].astype(float)

# Plot
plt.figure(figsize=(12,5))
plt.plot(height["datetime"], height["wse"], label = "Waterhoogte metingen") #height werkt in "datetime" en "wse" voor datum en hoogte
plt.title("Historische waterhoogte Brokopondomeer")
plt.xlabel("Datum")
plt.ylabel("Waterhoogte [m]")
plt.axhline(42.92, color = "r", linestyle='--', label = "Kritische hoogte")
plt.axhline(40.5, color = "r", label = "Zeer kritische hoogte")
plt.legend()
plt.grid()
plt.show();

In [ ]:
#begin en eind datum van beide metingen samen bepalen. 
start = max(opp["Date"].min(), height["datetime"].min())
end = min(opp["Date"].max(), height["datetime"].max())

In [ ]:
# oppervlakte tussen de start en einddatum zetten. 
opp = opp[
    (opp["Date"] >= start) &
    (opp["Date"] <= end)
].copy()

In [ ]:
# hoogte en oppervlakte metingen plotten. 
fig, ax1 = plt.subplots(figsize=(16,7))
ax1.plot(opp["Date"], opp["Value"], 'b', label = 'Oppervlakte')
ax1.set_xlabel("Datum")
ax1.set_ylabel("Oppervlakte stuwmeer [m^2]")
plt.legend(loc = 'lower left')

# Right y-axis: water level
ax2 = ax1.twinx()
ax2.plot(height["datetime"], height["wse"], 'g', label = 'Hoogte')
ax2.set_ylabel("Waterhoogte [m]")

plt.title("Brokopondomeer metingen waterhoogte en oppervlakte")
plt.legend(loc = 'lower right')
plt.grid()
plt.show()

In [ ]:
# Forcing genereren
basin_name = "Suriname"

# tijdsinterval
start_datum = "2019-01-01"
eind_datum = "2024-12-31"
start_datum = pd.to_datetime(start_datum, utc=True)
end_datum = pd.to_datetime(eind_datum, utc=True)
start_pd = start_datum.strftime("%Y-%m-%dT%H:%M:%SZ")
end_pd = end_datum.strftime("%Y-%m-%dT%H:%M:%SZ")
start_datum = start_pd
eind_datum = end_pd

# route naar shape file
shapefile = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "boven_suriname.shp"

HBV_model = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "hbv_bmi"

forcing_route = Path.home() / "BEP-Julian" / "BEP-Julian" / "Forcing" / "ERA5_SUR_2019_2024"/ "work" / "diagnostic" / "script" 

In [ ]:
# forcing genereren

# ERA5_forcing = ewatercycle.forcing.sources['LumpedMakkinkForcing'].generate(
#     dataset="ERA5",
#     start_time=start_datum,
#     end_time=eind_datum,
#     shape=shapefile,
#     directory=forcing_route
# )

# forcing inladen
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=forcing_route)

print(f"The forcing object you created: \n {ERA5_forcing}")

In [ ]:
# verdampingen en neerslag bruikbaar maken
evap = ERA5_forcing.to_xarray()["evspsblpot"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
prec = ERA5_forcing.to_xarray()["pr"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
fig, ax = plt.subplots(figsize=(12,5))

# evap.plot(ax=ax, label='Potential evaporation')
# prec.plot(ax=ax, label='Precipitation')
# ax.set_ylabel('mm/s')
# ax.set_xlabel("tijd")
# plt.title("E en P in mm/s, sampled per dag")
# ax.legend();

In [ ]:
# neerslag bruikbaar maken en plotten in mm/seconde
evap = ERA5_forcing.to_xarray()["evspsblpot"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
prec = ERA5_forcing.to_xarray()["pr"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
fig, ax = plt.subplots(figsize=(12,5))

# evap.plot(ax=ax, label='Potential evaporation')
# prec.plot(ax=ax, label='Precipitation')
# ax.set_ylabel('mm/s')
# ax.set_xlabel("tijd")
# plt.title("E en P in mm/s, sampled per dag")
# ax.legend();

In [ ]:
# neerslag bruikbaar maken en plotten in mm/dag
print(evap.min().values, evap.max().values)
print(prec.min().values, prec.max().values)
evap_dag = evap*3600*24
prec_dag = prec*3600*24
fig, ax = plt.subplots(figsize=(12,5))
evap_dag.plot(ax=ax, label="Evaporation [mm/day]")
prec_dag.plot(ax=ax, label="Precipitation [mm/day]")

# ax.set_ylabel("mm/dag")
# ax.set_xlabel("tijd")
# ax.legend()
# plt.title("E en P in mm/dag, sampled per dag")
# plt.show()

In [ ]:
# hoogte-volume verhouding plotten

fig, ax1 = plt.subplots(figsize=(5,6))
testH = np.arange(0, 55, 0.1)
testV = np.zeros(len(testH))
for i in range(len(testH)):
    testV[i] = (testH[i] - 36.715)/0.9096
plt.plot(testV, testH, label = "Rule curve")

testHh = np.arange(0, 55, 0.1)
testVv = np.zeros(len(testHh))

testVvv = np.arange(0,20, 0.03)
testHhh = np.zeros(len(testVvv))
for i in range(len(testVvv)):
    testHhh[i] = 8 + 15*np.log(testVvv[i]+2)
plt.plot(testVvv, testHhh, label = "Verhouding H en V")

plt.xlim(0, 20)
plt.title("Bathymetrie Brokopondomeer")
plt.legend(loc = 'lower right')
ax1.set_xlabel("Volume meer [km^3]")
ax1.set_ylabel("Hoogte meer [m]")
plt.grid();

In [ ]:
# hoogte-volume verhouding toepassen op hoogtemetingen

fig, ax1 = plt.subplots(figsize=(12,5))
H = height["wse"]
V = np.zeros(len(H))
# H = 8 + 15*np.log(V+2) met np.log = ln
# V = math.exp((H-8)/15)-2
for i in range(len(H)):
    V[i] = math.exp((H[i]-8)/15)-2
plt.plot(height["datetime"], V)
plt.title("V in km^3 according to (Sterl et al., 2020)")
plt.grid();

In [ ]:
# tijd tussen metingen plotten in dagen
fig, ax1 = plt.subplots(figsize=(12,5))
height["time_diff"] = height["datetime"].diff()
height["time_diff_hours"] = (
    height["datetime"].diff().dt.total_seconds() / (3600*24)
)
plt.plot(height["datetime"], height["time_diff_hours"])
plt.title("Tijd tussen hoogtemetingen Brokopondomeer in dagen")
plt.xlabel("Datum")
plt.ylabel("Dagen tussen metingen")
plt.grid();

In [ ]:
# verdamping en instroom model 1 berekenen en plotten

height["datetime_local_none_test"] = height["datetime"].dt.tz_localize(None)
fig, ax1 = plt.subplots(figsize=(16,5))

opp["Date"] = opp["Date"].dt.tz_localize(None)
begin_datum = pd.to_datetime(start_datum).tz_localize(None)
eind_datum = pd.to_datetime(eind_datum).tz_localize(None)
evap = evap.sel(time=slice(begin_datum, eind_datum))

height = height[
    (height["datetime_local_none_test"] >= begin_datum) &
    (height["datetime_local_none_test"] <= eind_datum)
].copy()

height = height.reset_index(drop=True)

height = height.reset_index(drop=True)

area_interp = np.interp(height["datetime_local_none_test"].astype("int64"), opp["Date"].astype("int64"), opp["Value"])

H = height["wse"].to_numpy()
V = np.exp((H - 8)/15) - 2
Vv = V * 10**9                                                            # van km^3 naar m^3
delta_s = np.zeros(len(height["datetime_local_none_test"]))
E = np.zeros(len(height["datetime_local_none_test"]))
opper = np.zeros(len(height["datetime_local_none_test"]))
Q_in_L = np.zeros(len(height["datetime_local_none_test"]))
E_v = np.zeros(len(height["datetime_local_none_test"]))
Q_in_L_sum = np.zeros(len(height["datetime_local_none_test"]))
Q_in_L_sum_test = np.zeros(len(height["datetime_local_none_test"]))
Q_out = 200

for i in range(len(height["datetime_local_none_test"])-1):
    
    t0 = height["datetime_local_none_test"].iloc[i]
    t1 = height["datetime_local_none_test"].iloc[i+1]

    dt_seconds = (t1 - t0).total_seconds()
    delta_s[i+1] = (Vv[i+1] - Vv[i]) / dt_seconds
    
    evap_interval = evap.sel(time=slice(t0, t1))
    E[i+1] = evap_interval.mean().values * 10**-3 #/dt_seconds       # van mm/s naar m/s

    A_t = area_interp[i+1]
    E_v[i+1] = A_t * 1e6 * E[i+1]

    Q_in_L[i+1] = delta_s[i+1] + E_v[i+1] + Q_out
    Q_in_L[i+1] = np.nan_to_num(Q_in_L[i+1], nan=0.0)
    Q_in_L_sum[i] = Q_in_L.sum()
    Q_in_L_sum_test = np.cumsum(Q_in_L)

plt.plot(height["datetime_local_none_test"], Q_in_L, label = "Q_in model 1 ")
plt.plot(height["datetime_local_none_test"], E_v, label = "Verdamping model 1")
plt.xlabel("Datum")
plt.ylabel("Discharge [m^3/s]")
plt.legend()
plt.title("Qin per dag m^3/s")
plt.grid();

In [ ]:
# functie om instroom meer naar hooogte om te zetten

def level(Qin, Qout, ERA5_forcing, A):
    #Qin is array met discharge van rivier in meer, gebaseerd op E en P en dagelijks
    #Qout is een standaard baseflow uit het meer
    #E is verdampingsdata van ERA5, dagelijks
    #A is oppervlakte meer in m^2 
    
    dt = 3600*24
    E = ERA5_forcing.to_xarray()["evspsblpot"] * dt
    L = np.zeros(len(E))
    
    for i in range(len(E)):
        dL = (Qin[i] - Qout - E[i]) / A
        L[i] = L[i-1] + dL*dt
    return L

In [ ]:
# model 2 

def discharge(parameters, forcing):
    # parameters zijn;
    #   Qgw - ground water flow in eht meer
    #   alpha - neerslag/verdamping coefficient
    #   beta - neerslag/verdamping coefficient
    ERA5_forcing = forcing

    E_da = ERA5_forcing.to_xarray()["evspsblpot"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
    P_da = ERA5_forcing.to_xarray()["pr"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag

    Qg, alpha, beta = parameters

    dt = 86400

    Qin = []

    for current_timestep in range(len(P_da)):

        P = P_da.isel(time=current_timestep).to_numpy() * dt
        E = E_da.isel(time=current_timestep).to_numpy() * dt

        Q = Qg + alpha * (P - E)**beta

        Qin.append(Q)

    return Qin # in m^3/s